In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.3 Quantization

Quantization stores weights in fewer bits, 8-bit or 4-bit instead of 32, cutting
memory by 4-8x. It is not free: low precision costs some accuracy. The rule is to
*measure* the degradation, never assume it.

In [ ]:
import copy
from pocketlm import train_tiny, pick_config, fake_quantize, estimate_loss

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
g = torch.Generator().manual_seed(0)
before = estimate_loss(model, data, model.cfg.block_size, 16, iters=10, generator=g)
qmodel = copy.deepcopy(model)
with torch.no_grad():
    for p in qmodel.parameters():
        p.copy_(fake_quantize(p, bits=4))
g2 = torch.Generator().manual_seed(0)
after = estimate_loss(qmodel, data, model.cfg.block_size, 16, iters=10, generator=g2)
print(f"loss fp32: {before:.3f}  ->  4-bit: {after:.3f}  (delta {after - before:+.3f})")

Four-bit weights take a quarter of the memory; the loss moves by the printed delta.
On real models 8-bit is nearly free and 4-bit is a deliberate trade, always backed
by a measured number like this one.

In [ ]:
import math
assert math.isfinite(after)
assert after > 0